# Pipeline Overview

DB health at a glance: posting counts by status, source breakdown, and what's waiting in each stage.

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import select, func, text
from sqlalchemy.orm import Session
from src.db import get_engine, Posting

engine = get_engine(Path('../data/jobs.db'))
pd.set_option('display.max_colwidth', 80)

## Status pipeline

In [ ]:
with Session(engine) as s:
    rows = s.execute(
        select(Posting.status, func.count().label('count'))
        .group_by(Posting.status)
        .order_by(func.count().desc())
    ).all()

STATUS_ORDER = ['new', 'ranked', 'matched', 'generated', 'reviewed', 'applied', 'skipped']
df_status = pd.DataFrame(rows, columns=['status', 'count'])
df_status['status'] = pd.Categorical(df_status['status'], categories=STATUS_ORDER, ordered=True)
df_status = df_status.sort_values('status')

colours = {
    'new': '#aaa', 'ranked': '#4c9be8', 'matched': '#5cb85c',
    'generated': '#f0ad4e', 'reviewed': '#9b59b6',
    'applied': '#2ecc71', 'skipped': '#e74c3c',
}

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.barh(
    df_status['status'], df_status['count'],
    color=[colours.get(s, '#aaa') for s in df_status['status']]
)
ax.bar_label(bars, padding=4)
ax.set_xlabel('Postings')
ax.set_title('Postings by pipeline status')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(df_status.to_string(index=False))

## Source breakdown

In [ ]:
with Session(engine) as s:
    rows = s.execute(
        select(Posting.source, Posting.company, func.count().label('n'))
        .group_by(Posting.source, Posting.company)
        .order_by(func.count().desc())
    ).all()

df_src = pd.DataFrame(rows, columns=['source', 'company', 'postings'])
print(df_src.to_string(index=False))

## What's waiting at each stage

In [ ]:
for stage in ['new', 'ranked', 'matched', 'generated']:
    with Session(engine) as s:
        n = s.scalar(select(func.count()).where(Posting.status == stage))
    print(f'{stage:12s}: {n} posting(s) ready to advance')

## Recently added postings

In [ ]:
with Session(engine) as s:
    recent = s.scalars(
        select(Posting).order_by(Posting.created_at.desc()).limit(10)
    ).all()

df_recent = pd.DataFrame([
    {'created': p.created_at, 'company': p.company, 'title': p.title, 'status': p.status}
    for p in recent
])
df_recent